In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from ensemble import compute_ensemble
import sys
sys.path.append('../')
from aux_func import get_data
from analysis import * 

import matplotlib as mpl
from mosqlient.prediction_optimize import get_df_pars_ls
from harmonize_preds import aux_functions

# Definir a cor das bordas (spines) como cinza
mpl.rcParams['axes.edgecolor'] = 'gray'

# Definir a cor das linhas dos ticks maiores e menores como cinza
mpl.rcParams['xtick.color'] = 'gray'
mpl.rcParams['ytick.color'] = 'gray'
mpl.rcParams['xtick.labelcolor'] = 'black'
mpl.rcParams['ytick.labelcolor'] = 'black'
plt.rcParams['axes.labelsize'] = 16 # Axis labels
plt.rcParams['xtick.labelsize'] = 14  # X-axis tick labels
plt.rcParams['ytick.labelsize'] = 14  # Y-axis tick labels
plt.rcParams['font.size'] = 16  # General font size
FONT = 12

Challenge: 

In [2]:
challenge = 'chik_state'
col_region = 'adm_1'

Load data: 

In [3]:
df_dengue_state = get_data(challenge)
df_dengue_state.date = pd.to_datetime(df_dengue_state.date)
df_dengue_state.head()

,date,adm_1,casos
0,2013-12-29,11,0
1,2013-12-29,12,0
2,2013-12-29,13,0
3,2013-12-29,14,0
4,2013-12-29,15,0


Load ranking: 

In [4]:
remove_models = ['3rd_imdc_-unifesp-_-4mosqueteiras-',
                                      '3rd_imdc_fiocruz_zerolags',
                                        '3rd_imdc_rki_rki_zki_ph',
                               '3rd_imdc_rki_rki_zki_ph_lstm_geo',
                                         '3rd_imdc_unesp_recogna',
 '3rd_imdc_universidad_del_valle_grupo_modelamiento_datos_dengue',
 '3rd_imdc_unesp_recogna_daniel']

In [5]:
df_rank = pd.read_csv(f'../predictions/rank_ratio_{challenge}.csv')

model_baseline = '3rd_imdc_procc_bb_model'

df_wis_base = df_rank.loc[df_rank.model == model_baseline]

df_rank = df_rank.loc[df_rank.model != model_baseline]

df_rank = df_rank.loc[~df_rank.model.isin(remove_models)]

df_rank = df_rank.loc[df_rank[col_region] != 32]

df_rank = df_rank.groupby([col_region, 
                                                                                                       "model"], as_index=False).agg(
                                        geometric_mean_ratio=("wis_ratio", lambda x: np.exp(np.mean(np.log(x)))),
                                    ).sort_values([col_region, "geometric_mean_ratio"])

df_rank.head()

,adm_1,model,geometric_mean_ratio
7,11,3rd_imdc_nus_nus-cerm,1.036956
8,11,3rd_imdc_purdue_neuralearth,1.068451
2,11,3rd_imdc_emap_lstm,1.079902
9,11,DS-OKSTATE-2026,1.149659
0,11,3rd_imdc_emap_epidematicos_prophet_fixed,1.156044


In [6]:
df_rank.model.unique().shape

(11,)

Load preds: 

In [7]:
df_preds = pd.read_csv(f'../predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds = df_preds.loc[~df_preds.model.isin(remove_models)]
df_preds = df_preds.loc[df_preds.model != model_baseline]


df_preds = df_preds.loc[df_preds[col_region] != 32]

df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,wis,model
0,2022-10-09,2.743512,3.561896,5.283421,9.016248,17.670073,40.444549,89.679029,143.716210,206.231430,33,5354,1,82.77,3rd_imdc_emap_lstm
1,2022-10-16,3.743952,5.428969,6.920945,12.201003,27.237822,55.050853,112.596736,216.009888,325.011995,33,5354,1,82.77,3rd_imdc_emap_lstm
2,2022-10-23,4.875423,5.693745,8.176445,12.618417,22.377075,42.060248,77.746351,114.612724,173.066908,33,5354,1,82.77,3rd_imdc_emap_lstm
3,2022-10-30,3.250737,4.192324,5.602107,9.793079,17.521617,34.101676,68.882955,117.519627,166.359700,33,5354,1,82.77,3rd_imdc_emap_lstm
4,2022-11-06,2.961620,3.732665,5.172567,9.438648,20.027886,46.905846,103.815092,215.222367,336.458355,33,5354,1,82.77,3rd_imdc_emap_lstm


In [8]:
df_preds.model.unique().shape

(11,)

Get best models: 

In [9]:
n_models = 10
best_models = {}

for state in df_rank[col_region].unique():

    best_models[f'{state}'] = df_rank.loc[(df_rank[col_region] == state)].sort_values(by = 'geometric_mean_ratio').model.values[:n_models]


In [10]:
best_models['41']

<ArrowStringArray>
[                'mosqlimate-ai-competitor',
              '3rd_imdc_purdue_neuralearth',
  '3rd_imdc_lncc_surge_model26_chikungunya',
                       '3rd_imdc_emap_lstm',
 '3rd_imdc_emap_epidematicos_prophet_fixed',
                  '3rd_imdc_emap_xgbsillas',
                    '3rd_imdc_nus_nus-cerm',
      '3rd_imdc_lncc_clidengo26chikungunya',
                          'DS-OKSTATE-2026',
          '3rd_imdc_lncc_ARp26_chikungunya']
Length: 10, dtype: str

Ensemble of the median of the top K models:

Load forecasts:

In [11]:
df_for = pd.read_csv(f'../predictions/forecast_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_for.date = pd.to_datetime(df_for.date)
df_for = df_for.loc[~df_for.model.isin(remove_models)]
df_for = df_for.loc[df_for.model != model_baseline]


df_for = df_for.loc[df_for[col_region] != 32]

df_for.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,model
0,2026-10-11,0.200655,0.519182,0.965134,2.225290,3.166690,5.676564,8.080002,9.462498,11.961388,12,13308,Forecast,3rd_imdc_emap_epidematicos_sarimax_fixed
1,2026-10-18,0.000000,0.084189,0.517639,1.488684,2.805839,4.860114,8.043318,10.547137,14.231962,12,13308,Forecast,3rd_imdc_emap_epidematicos_sarimax_fixed
2,2026-10-25,0.000000,0.000532,0.309755,1.096067,2.452235,5.063127,8.367789,12.152720,16.359749,12,13308,Forecast,3rd_imdc_emap_epidematicos_sarimax_fixed
3,2026-11-01,0.000000,0.000000,0.049297,0.803332,2.283912,4.719045,8.782080,13.205769,17.227440,12,13308,Forecast,3rd_imdc_emap_epidematicos_sarimax_fixed
4,2026-11-08,0.000000,0.000000,0.000000,0.620797,2.063121,4.641258,8.815452,12.637288,16.683337,12,13308,Forecast,3rd_imdc_emap_epidematicos_sarimax_fixed


In [12]:
df_for.model.unique().shape

(11,)

In [13]:
np.setdiff1d(df_for.model.unique(), df_preds.model.unique())

array([], dtype=object)

In [14]:
np.setdiff1d( df_preds.model.unique(), df_for.model.unique())

array([], dtype=object)

In [15]:
dfm_top = pd.DataFrame()

for st in df_for[col_region].unique(): 
    df_median_ens = compute_ensemble(df_for.loc[(df_for[col_region] == st) &
                                                  (df_for.model.isin(best_models[f'{st}']))])
    df_median_ens.head()

    dfm_top = pd.concat([dfm_top, df_median_ens], ignore_index = True)

dfm_top['model_id'] = 'E_median_top'

dfm_top.head()

,date,adm_1,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,model_id
0,2026-10-11,12,0.014015,0.071635,0.097095,0.401176,1.385212,3.019548,4.567273,7.744528,9.424722,E_median_top
1,2026-10-18,12,0.000000,0.062214,0.108057,0.451789,1.429170,3.218501,4.606893,7.038406,9.924812,E_median_top
2,2026-10-25,12,0.000000,0.019740,0.125938,0.490521,1.446291,3.709993,5.270509,6.387104,8.845027,E_median_top
3,2026-11-01,12,0.000000,0.017377,0.057868,0.420702,1.217666,3.712967,5.179566,7.285387,9.937315,E_median_top
4,2026-11-08,12,0.000000,0.015651,0.054853,0.461714,1.238348,3.329854,4.746622,7.922177,11.586268,E_median_top


In [16]:
dfm_top.to_csv(f'../predictions/ensemble_weekly_2026_top_10_{challenge}.csv.gz', index = False)

Ensemble of the last season: 

In [17]:
dfm_top_2025 = pd.DataFrame()

for st in df_preds[col_region].unique(): 
    df_median_ens = compute_ensemble(df_preds.loc[(df_preds[col_region] == st) & (df_preds.validation == 4) &
                                                  (df_preds.model.isin(best_models[f'{st}']))])
    df_median_ens.head()

    dfm_top_2025 = pd.concat([dfm_top_2025, df_median_ens], ignore_index = True)

dfm_top_2025['model_id'] = 'E_median_top'

dfm_top_2025.head()

,date,adm_1,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,model_id
0,2025-10-05,33,0.630623,1.198377,1.814893,6.225253,18.304680,33.988933,52.693038,84.795422,128.500000,E_median_top
1,2025-10-12,33,0.379597,0.718747,1.496209,5.897110,18.600864,34.856363,62.436843,90.797684,151.500000,E_median_top
2,2025-10-19,33,0.536955,0.959016,1.705436,6.703098,19.796668,41.522605,62.789537,87.001150,180.009894,E_median_top
3,2025-10-26,33,0.602318,1.111967,1.867768,7.820277,22.541050,47.878050,87.874395,111.204584,192.544609,E_median_top
4,2025-11-02,33,1.299467,2.082938,2.954034,8.994545,24.207946,45.861361,84.509503,116.875932,194.392213,E_median_top


In [18]:
dfm_top_2025.date.max()

Timestamp('2026-10-04 00:00:00')

In [19]:
dfm_top_2025.to_csv(f'../predictions/ensemble_weekly_2025_top_10_{challenge}.csv.gz', index = False)

Compute rho to get cumulative cases: 

In [20]:
def get_total_cases(df_preds, df_for,  df_dengue_state, col_region, region, n_paths=1000, seed=0):
    """
    Processa a simulação e métricas para uma única combinação de modelo e validação.
    """
    rng = np.random.default_rng(seed)
    
    # 1. Filtragem e preparação de parâmetros
    df_preds_m = df_preds.loc[(df_preds[col_region]== region)]
    
    df_for_m = df_for.loc[(df_for[col_region]== region)]

    #print(df_preds_m.shape)
    if df_preds_m.empty:
        return None  # Retorna None se não houver dados para essa combinação
        
    df_pars = get_df_pars_ls(df_preds_m)
    #df_pars.loc[(df_pars.mu <= 0) | (df_pars.mu.isna()), 'mu'] = 0.1
    df_pars.loc[(df_pars.sigma <= 0) | (df_pars.sigma.isna()), 'sigma'] = 0.1

    df_pars_for = get_df_pars_ls(df_for_m)
    #df_pars_for.loc[(df_pars_for.mu <= 0) | (df_pars_for.mu.isna()), 'mu'] = 0.1
    df_pars_for.loc[(df_pars_for.sigma <= 0) | (df_pars_for.sigma.isna()), 'sigma'] = 0.1

    # 2. Estimativa de Rho e Marginais
    df_rho_input = df_pars.merge(
        df_dengue_state, on=['date', col_region,], how='left'
    )

    df_rho_input = df_rho_input.dropna()
    
    #print(df_rho_input)

    rho = estimate_rho(df_rho_input)

    #print(rho)

    marginals_uf = build_marginals(df_pars_for, col_region, region, None)

    # 3. Geração das trajetórias (paths)
    paths = np.vstack([
        sample_path(marginals_uf, rho, random_state=rng)
        for _ in range(n_paths)
    ])

    #print(paths.shape)

    #plt.plot(paths[1, : ])

    # 4. Cálculo das métricas baseadas em todas as trajetórias
    season_total = paths.sum(axis=1)

    #print(season_total.shape)


    # 6. Consolidação dos resultados em um dicionário de intervalos
    metrics = {
        'model': 'ensemble',
        'validation': 'Forecast',
        col_region: region
    }
    
    # Dicionário mapeando o nome da métrica para o seu array de distribuição
    distributions = {
        'season_total': season_total,
    }
    
    # Calcula os quantis automaticamente para cada métrica
    for name, dist in distributions.items():
        lo, med, hi = np.quantile(dist, [0.05, 0.5, 0.95])
        metrics[f'{name}_p5'] = lo
        metrics[f'{name}_p50'] = med
        metrics[f'{name}_p95'] = hi

    return metrics

In [21]:
df_cum_cases = pd.DataFrame()

for adm_1 in df_for.adm_1.unique(): 

    metrics = get_total_cases(dfm_top_2025, dfm_top,  df_dengue_state, col_region, adm_1, n_paths=1000, seed=0)

    df_cum_cases = pd.concat([ df_cum_cases, pd.DataFrame([metrics])], ignore_index= True)

df_cum_cases = df_cum_cases.rename(columns = {'season_total_p5': 'lower_90', 
                                              'season_total_p50': 'pred', 
                                              'season_total_p95': 'upper_90'})
df_cum_cases.head()

,model,validation,adm_1,lower_90,pred,upper_90
0,ensemble,Forecast,12,76.883098,158.878725,396.910270
1,ensemble,Forecast,27,592.270393,1674.473300,5317.670913
2,ensemble,Forecast,13,109.265234,192.582758,383.759215
3,ensemble,Forecast,16,59.375051,129.123079,388.597498
4,ensemble,Forecast,29,2404.571198,8762.198472,37368.241629


In [22]:
df_cum_cases.to_csv(f'../predictions/preds_ensemble_2026_cum_cases_{challenge}.csv.gz', index = False)